# Détection des Zones d'Altération Hydrothermale via Indices Spectraux

## Introduction
Ce notebook explore la détection des minéraux d'altération hydrothermale (argiles, oxydes de fer, carbonates) essentiels pour l'exploration minière. Par l'analyse des propriétés d'absorption dans le spectre Infrarouge à ondes courtes (SWIR), nous pouvons localiser les zones où les fluides minéralisateurs ont modifié la roche encaissante.

## Objectifs
*   **Calcul d'indices minéralogiques** : Utiliser les ratios de bandes Landsat 8/Sentinel-2 pour isoler les argiles et le fer.
*   **Identification des cibles minières** : Synthétiser ces indices pour créer une carte de probabilité d'altération.
*   **Validation spatiale** : Visualiser les anomalies sur la zone d'étude au Congo.

## Méthodologie
1.  **Setup** : Installation des outils géospatiaux.
2.  **Acquisition** : Téléchargement d'images Landsat 8 (idéales pour le minéralogique grâce aux bandes SWIR précises).
3.  **Calcul Expert** : Application des formules de ratios de bandes (Clay Index, Iron Index).
4.  **Visualisation** : Cartographie des anomalies à forte intensité.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration et Initialisation
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib seaborn -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ GEE Initialisé')

## Zone d'Étude (ROI)
Vérification de la localisation géographique pour l'analyse minéralogique.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données Landsat 8
Landsat 8 est privilégié ici pour ses bandes SWIR (B6 et B7) particulièrement sensibles aux minéraux hydroxylés (argiles).

In [ ]:
# ====================================================
# ÉTAPE 3 : Acquisition satellite
# ====================================================
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
              .filterBounds(roi)
              .filterDate('2022-01-01', '2023-12-31')
              .median().clip(roi))

bands = ['SR_B2', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
image = collection.select(bands)
geemap.ee_export_image(image, 'minerals.tif', scale=30, region=roi)
print('✅ Image exportée pour calcul d'indices')

## Calcul des Indices d'Altération
Nous calculons le Clay Index (Ratio SWIR1/SWIR2) et l'Iron Oxide Index (Ratio Rouge/Bleu). Les zones où ces deux indices sont élevés simultanément indiquent une forte probabilité d'altération hydrothermale.

In [ ]:
# ====================================================
# ÉTAPE 4 : Calcul Minéralogique
# ====================================================
with rasterio.open('minerals.tif') as src:
    data = src.read().astype(np.float32)

blue, red, nir, swir1, swir2 = data[0], data[1], data[2], data[3], data[4]

clay_index = swir1 / (swir2 + 1e-6)
iron_index = red / (blue + 1e-6)
alteration_score = (clay_index * iron_index)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(clay_index, cmap='YlOrBr')
plt.title('Indice des Argiles')
plt.subplot(1, 2, 2)
plt.imshow(alteration_score, cmap='magma')
plt.title('Zones d'Altération Potentielles')
plt.show()